# Advanced packages - Processing output

Part of the **synthetic-valley advanced-packages** series. This notebook runs the shared advanced model in `models/` - whatever advanced packages the other notebooks have built into it - and plots the results against the calibration targets.

- **UZF** (`mf6-adv-uzf`) - recharge (RCH) -> Unsaturated Zone Flow
- **MAW** (`mf6-adv-maw`) - wells (WEL) -> Multi-Aquifer Well
- **SFR** (`mf6-adv-sfr`) - river (RIV) -> Streamflow Routing
- **LAK** (`mf6-adv-lak`) - high-K lake -> Lake package
- **MVR** (`mf6-adv-mvr`) - Water Mover (requires UZF, LAK, SFR)
- **Processing output** (`mf6-adv-processing`) - run the model and plot results *(this notebook)*

**Run order.** If you run this notebook before building any advanced packages, `load_or_create_advanced_model` copies the calibrated base model into `models/` and this notebook runs and post-processes that base model. As you build UZF, MAW, SFR, LAK, and MVR with the other notebooks, re-running this notebook reflects whichever packages are present: the post-processing detects the SFR and LAK packages and switches the surface-water budget term and streamflow/lake plots accordingly.

## Imports and setup

Import FloPy and the helpers, then choose the temporal resolution (`sample_frequency`) and the model name.

In [ ]:
%matplotlib inline
import pathlib as pl
import pickle

import flopy
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import numpy as np
import pandas as pd
from mf6_notebook_helpers import (
    load_or_create_advanced_model,
    load_spatial_data,
    load_temporal_data,
    synthetic_valley_workspaces,
)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

## Load and run the advanced model

Load the shared advanced model from `models/` (or create it from the base model on first use), run it, and read the grid dimensions. Then detect which advanced packages are present so the plots below adapt to the current state of the model.

In [ ]:
sim = load_or_create_advanced_model(sample_frequency, name)
gwf = sim.get_model(name)

success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

nlay, nrow, ncol = gwf.dis.nlay.array, gwf.dis.nrow.array, gwf.dis.ncol.array
shape2d = (nrow, ncol)
_, advanced_ws = synthetic_valley_workspaces(sample_frequency, name)

In [ ]:
# detect the advanced packages that have been built into the model; the plots
# below choose what to draw from these instead of hard-coded toggles
has_sfr = gwf.get_package("SFR-1") is not None
has_lak = gwf.get_package("LAK-1") is not None
print(f"SFR present: {has_sfr}, LAK present: {has_lak}")
print("packages:", gwf.get_package_list())

### Load the supporting data

Read the "truth" dataset (for the lake footprint and observed heads) and the temporal forcing, then compute the elapsed time at the end of the calibration period and the two transient prediction periods.

In [ ]:
nc_ds, lake_location, lake_area = load_spatial_data()
temporal_df = load_temporal_data(sample_frequency)

In [ ]:
idx_end_calibration = 0
if sample_frequency == "monthly":
    idx_end_period2 = 120
    idx_end_period3 = 240
elif sample_frequency == "annual":
    idx_end_period2 = 10
    idx_end_period3 = 20
else:
    raise ValueError(f"invalid sample_frequency: '{sample_frequency}'")

start_date = pd.to_datetime("1962-01-01 00:00:00")

end_calibration = temporal_df.index[idx_end_calibration]
end_period_two = temporal_df.index[idx_end_period2]
end_period_three = temporal_df.index[idx_end_period3]

totim_end = [float((end_calibration - start_date).days)]
totim_end += [float((end_period_two - start_date).days)]
totim_end += [float((end_period_three - start_date).days)]
totim_end

## Plot the results

Visualize the simulated results and compare them with the calibration targets.

### Model properties

Maps of hydraulic conductivity, bottom elevation, and cell thickness by layer. Define `plot_layers()` once here — it draws a `(nlay, nrow, ncol)` array as a row of by-layer maps in the USGS map style (with labeled white contours when `levels` is given) — and reuse it for every by-layer map below.

In [ ]:
def plot_layers(array, title, levels=None, masked_values=None):
    """Map a (nlay, nrow, ncol) array layer by layer in a row of USGS-style map
    axes; overlay labeled white contours when `levels` is given."""
    with flopy.plot.styles.USGSMap():
        fig, axs = plt.subplots(1, nlay, figsize=(9, 3), sharey=True)
        fig.suptitle(title)
        for idx in range(nlay):
            ax = axs[idx]
            mm = flopy.plot.PlotMapView(
                model=gwf,
                layer=idx,
                ax=ax,
            )
            mm.plot_array(array, masked_values=masked_values)
            if levels is not None:
                cs = mm.contour_array(array, colors="white", levels=levels)
                plt.clabel(cs, inline=True, fontsize=8)
            ax.set_title(f"Layer {idx + 1}")

In [ ]:
plot_layers(gwf.npf.k.array, "Hydraulic conductivity", masked_values=[2000000.0])

In [ ]:
plot_layers(gwf.dis.botm.array, "Bottom Elevation")

In [ ]:
plot_layers(gwf.modelgrid.cell_thickness, "Cell thickness")

**What to look for.** These three maps show the model's fixed properties by layer, before anything is simulated: hydraulic conductivity `K` (how easily water moves - the very high lake cells are masked out), the bottom elevation of each cell, and the cell thickness. They set the stage for the head and flow results below.

### Simulated heads and drawdown

A helper to retrieve heads at a given time (substituting lake stage in the lake cells when the LAK package is present), followed by the head and drawdown maps for each period.

In [ ]:
def get_heads(totim):
    hds = gwf.output.head().get_data(totim=totim)
    if has_lak:
        stage = gwf.lak.output.stage().get_data(totim=totim)
        h1 = hds[0]
        stage = np.full(shape2d, stage[0], dtype=float)
        idx = lake_location == 1
        h1[idx] = stage[idx]
        hds[0, :, :] = h1
    return hds

In [ ]:
levels = np.arange(2, 20.0, 2)

#### Calibration

Simulated heads at the end of the calibration period, compared with the calibration targets.

In [ ]:
totim = totim_end[0]
hds = get_heads(totim)

In [ ]:
v = nc_ds["head_layer2"].values
v.min(), v.max(), v.mean()

In [ ]:
for k in range(nlay):
    diff = hds[k] - nc_ds[f"head_layer{k + 1}"].values
    print(f"{k + 1}: {diff.min()} {diff.max()} {diff.mean()}")

In [ ]:
plot_layers(hds, "Heads - Calibration", levels=levels)

**What to look for.** These are the simulated heads at the end of the calibration period, contoured by layer. Compare them with the layer-by-layer differences from the "truth" dataset printed just above: a well-calibrated model reproduces the observed head field closely. The residual maps a few cells down quantify where the fit is high or low.

**Inspecting the budget.** The cell below lists the flow terms MODFLOW saved in the cell-by-cell budget file and then picks the right label for the surface-water exchange - `"sfr"` when the SFR package is active, `"riv"` otherwise - so the post-processing below works no matter which packages are present.

In [ ]:
# list the flow terms saved in the cell-by-cell budget file, then choose the
# label for the surface-water exchange: "sfr" when the SFR package is active,
# "riv" otherwise
budget = gwf.output.budget()
record_names = [
    r.strip() for r in np.array(budget.get_unique_record_names()).astype(str)
]
print("Available budget terms:", record_names)

riv_text = "sfr" if has_sfr else "riv"
v = budget.get_data(text=riv_text, totim=totim)[0]["q"]
print(f"\nUsing '{riv_text}' for the surface-water exchange term")
print(f"River infiltration everywhere positive: {np.all(v > 0)}\n{v}")

##### Calculate the residuals

Compute the difference between simulated and observed heads at the water-table and lower-aquifer observation wells, and summarize the error statistics.

In [ ]:
obs_path = pl.Path("../data/synthetic-valley/data")
with open(obs_path / "obs_data.pkl", "rb") as f:
    obs_rc_locs, well_depth, aq_layer = pickle.load(f)

cal_loc_wt = [(0, i, j) for i, j in obs_rc_locs]
cal_loc_aq = [(aq_layer[idx], i, j) for idx, (i, j) in enumerate(obs_rc_locs)]

In [ ]:
botm = gwf.dis.botm.array
wt_obs = []
aq_layer = []
aq_obs = []
for idx, (i, j) in enumerate(obs_rc_locs):
    iloc = (i, j)
    tag = "head_layer1"
    wt_obs.append(float(nc_ds[tag].values[iloc]))
    wz = well_depth[idx]
    zcell = np.array(botm)[:, i, j]
    klay = 0
    for kk in range(1, nlay):
        z0 = zcell[kk - 1]
        z1 = zcell[kk]
        if wz < z0 and wz >= z1:
            klay = kk
            break
    tag = f"head_layer{klay + 1}"
    aq_layer.append(klay)
    aq_obs.append(float(nc_ds[tag].values[iloc]))

In [ ]:
sim_wt = np.array([hds[idx] for idx in cal_loc_wt])
resid_wt = sim_wt - np.array(wt_obs)
sim_aq = np.array([hds[idx] for idx in cal_loc_aq])
resid_aq = sim_aq - np.array(aq_obs)
resid_gb = np.concatenate((resid_wt, resid_aq))

In [ ]:
print(
    f"Water Table Statistics\nMean Error: {resid_wt.mean()} ft.\nRMSE:       {np.sqrt((resid_wt**2).sum()) / resid_wt.shape[0]} ft."
)
print(
    f"Lower Aquifer Statistics\nMean Error: {resid_aq.mean()} ft.\nRMSE:       {np.sqrt((resid_aq**2).sum()) / resid_aq.shape[0]} ft."
)
print(
    f"Global Statistics\nMean Error: {resid_gb.mean()} ft.\nRMSE:       {np.sqrt((resid_gb**2).sum()) / resid_gb.shape[0]} ft."
)

##### Plot the residuals

Interpolate and contour the head residuals to check for spatial bias in the calibration.

In [ ]:
xy = [
    (float(gwf.modelgrid.xcellcenters[i, j]), float(gwf.modelgrid.ycellcenters[i, j]))
    for i, j in obs_rc_locs
]
x, y = np.array(xy)[:, 0], np.array(xy)[:, 1]
grid_x, grid_y = np.meshgrid(gwf.modelgrid.xycenters[0], gwf.modelgrid.xycenters[1])

# Linearly interpolate the residuals (x, y) onto the grid
triang = tri.Triangulation(x, y)
interpolator = tri.LinearTriInterpolator(triang, resid_wt)
grid_resid_wt = interpolator(grid_x, grid_y)
interpolator = tri.LinearTriInterpolator(triang, resid_aq)
grid_resid_aq = interpolator(grid_x, grid_y)

resid_levels = np.arange(-2, 2.25, 0.25)

In [ ]:
with flopy.plot.styles.USGSMap():
    fig, axs = plt.subplots(1, 2, figsize=(8, 5), sharey=True)
    fig.suptitle("Residuals")

    ax = axs[0]
    ax.set_xlim(0, 12500)
    ax.set_ylim(0, 20000)
    mm = flopy.plot.PlotMapView(
        model=gwf,
        ax=ax,
        extent=gwf.modelgrid.extent,
    )
    mm.plot_array(lake_location, cmap="Blues_r", masked_values=[0])
    mm.plot_grid(lw=0.5, color="0.5")
    ax.scatter(x, y, s=3, c="black")
    for i, txt in enumerate(resid_wt):
        ax.annotate(f"{txt:.2f}", (x[i], y[i]))
    cs = ax.contour(
        grid_x,
        grid_y,
        grid_resid_wt,
        levels=resid_levels,
        linewidths=0.75,
        colors="red",
    )
    plt.clabel(cs, inline=True, fontsize=8)
    ax.set_title("Water Table")

    ax = axs[1]
    ax.set_xlim(0, 12500)
    ax.set_ylim(0, 20000)
    mm = flopy.plot.PlotMapView(
        model=gwf,
        ax=ax,
        extent=gwf.modelgrid.extent,
    )
    mm.plot_grid(lw=0.5, color="0.5")
    ax.scatter(x, y, s=3, c="black")
    for i, txt in enumerate(resid_aq):
        ax.annotate(f"{txt:.2f}", (x[i], y[i]), clip_on=False)
    cs = ax.contour(
        grid_x,
        grid_y,
        grid_resid_aq,
        levels=resid_levels,
        linewidths=0.75,
        colors="red",
    )
    plt.clabel(cs, inline=True, fontsize=8)
    ax.set_title("Lower Aquifer")

**NOTE:** There is spatial bias in the simulated results (*i.e.*, residuals are positive in the Northeast and negative in the Southwest).

#### First 10-year transient period

Simulated heads and drawdown at the end of the first transient prediction period.

In [ ]:
totim = totim_end[0]
totim1 = totim_end[1]

In [ ]:
hds = get_heads(totim1)
plot_layers(hds, "Heads - Transient Period 1", levels=levels)

In [ ]:
ddn = get_heads(totim) - get_heads(totim1)
plot_layers(ddn, "Drawdown - Transient Period 1", levels=levels)

ddn_max = ddn[:, 16, :].max()
print(f"Maximum Drawdown: {ddn_max}")
v = gwf.output.budget().get_data(text=riv_text, totim=totim1)[0]["q"]
print(f"Induced river infiltration: {np.all(v > 0)}\n{v}")

**What to look for - drawdown.** "Drawdown" here is the calibration-period head minus the later head, so positive values mark places where the water table has dropped. The cone of drawdown is centered on the pumping wells and is deepest in the pumped layers. The *induced river infiltration* check confirms that lowering heads near the stream draws additional water from it - the stream is helping to supply the wells.

#### Second 10-year transient period

Simulated heads and drawdown at the end of the second transient prediction period.

In [ ]:
totim2 = totim_end[2]

In [ ]:
hds = get_heads(totim2)
plot_layers(hds, "Heads - Transient Period 2 - Prediction", levels=levels)

In [ ]:
ddn = get_heads(totim) - get_heads(totim2)
plot_layers(ddn, "Drawdown - Transient Period 2 - Prediction", levels=levels)

ddn_max = ddn[:, 16, :].max()
print(f"Maximum Drawdown: {ddn_max}")
v = gwf.output.budget().get_data(text=riv_text, totim=totim2)[0]["q"]
print(f"Induced river infiltration: {np.all(v > 0)}\n{v}")

### Streamflow results

Streamflow at the southern boundary gauge and the percent reduction in discharge caused by pumping (drawn from the SFR gauge when the SFR package is present, otherwise the RIV observations).

In [ ]:
if has_sfr:
    df = gwf.sfr.output.obs().get_dataframe(start_datetime=start_date)
    df["RIV-FLOW"] /= -86400
else:
    df = gwf.riv.output.obs().get_dataframe()

df["RIV-SWGW"] /= -86400
df["TOTAL"] = df["RIV-SWGW"]

Q0 = df["TOTAL"].iloc[0]
df["PCT_DIFF"] = -100.0 * (df["TOTAL"] - Q0) / Q0
df

In [ ]:
if has_sfr:
    with flopy.plot.styles.USGSPlot():
        fig, axs = plt.subplots(2, 1, figsize=(9, 3), sharex=True)

        fig.suptitle("Southern Boundary - Gauge 1")

        ax = axs[0]
        ax.set_ylim(-5, 25)
        df["RIV-FLOW"].plot(ax=ax, ls="-", marker="o", clip_on=False)
        ax.axhline(0, lw=0.5, color="black")
        ax.set_ylabel("River\nDischarge, cfs")

        ax = axs[1]
        ax.set_ylim(-100, 100)
        df["PCT_DIFF"].plot(ax=ax, ls="-", marker="o", clip_on=False)
        ax.axhline(0, lw=0.5, color="black")
        ax.set_ylabel("Reduction\n in River\nDischarge, %")
        ax.set_xlabel("Stress Period")

**What to look for.** The top panel is discharge at the southern gauge; the bottom panel is its percent reduction relative to the first stress period. As pumping continues, gauge discharge falls and the percent reduction climbs - this is **streamflow capture**, the share of pumped water that is taken from the stream rather than from aquifer storage. (This figure is drawn only when the SFR package is present.)

### Lake stage

Simulated lake stage through time. Lake stage is written to the same observation file whether or not the LAK package is present (the LAK package writes it when present; the base model's GWF-OBS package writes the head in the high-K lake cell otherwise), so the plot works either way.

In [ ]:
fpth = advanced_ws / f"{name}.lake.obs.csv"
lake_df = flopy.utils.Mf6Obs(
    fpth,
).get_dataframe(start_datetime=start_date)
if has_lak:
    lake_df["LAKE-SWGW"] *= 12.0 / lake_area
lake_df

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(1, 1, figsize=(9, 1.5))

    lake_df["LAKE-STAGE"].plot(ax=ax, ls="-", marker="o", clip_on=False)
    ax.axhline(0, lw=0.5, color="black")
    ax.set_ylabel("Lake\nStage, ft")
    ax.set_xlabel("Stress Period")
    ax.set_ylim(10, 15)

**What to look for.** The lake stage rises and falls over the simulation as recharge, evaporation, and exchange with the aquifer change through time. When the LAK package is present this is the simulated lake water level; in the base model it is the head in the high-conductivity lake cell, so the plot works either way.

**Recap.** Starting from a calibrated base model, the advanced-packages series replaced its simple boundary conditions with advanced packages - `UZF` for recharge, `MAW` for wells, `SFR` for the river, and `LAK` for the lake - and connected them with the `MVR` mover, each built in its own notebook against the shared model in `models/`. This notebook ran that model and evaluated it against calibration targets and across two prediction periods, looking at head residuals, pumping-induced drawdown, streamflow capture, and lake stage. Because the plots draw only the packages actually present, the same notebook works whether you have built every advanced package or none.